# RAG With Arxiv

## Loading Necessary Keys

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable
groq_api_key=os.getenv("GROQ_API_KEY")

## Data ingestion (PDF)

In [ ]:
#PDFLoader
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('budget_speech.pdf')
docs=loader.load()
docs

1

## Text Splitting 

In [16]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_documents = text_splitter.split_documents(docs)

## Text Embeddings

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\vikram.vadhirajan\AppData\Local\Temp\ipykernel_31728\23041323.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
c:\14_Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\14_Langchain\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default.

## Vector Database creation and Retrives Creation

In [ ]:
from langchain.vectorstores import FAISS

db = FAISS.from_documents(
    split_documents,
    embedding_model
)

retriever = db.as_retriever(
    search_kwargs={"k": 4}
)

## LLM Model Creation

In [19]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000258F08F6BA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000258F0A08AD0>, model_name='openai/gpt-oss-120b', groq_api_key=SecretStr('**********'))

## Chat Prompt template Creation for understanding context

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


template = """
You are an AI assistant that answers questions using the provided context.

Context:
{context}

Question:
{question}

Provide a clear and concise answer.
"""

prompt = ChatPromptTemplate.from_template(template)

In [21]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

## Question and Answer

In [25]:
question = "What is Attention?"

response = rag_chain.invoke(question)

print(response)

**Attention** is a mechanism that maps a *query* vector and a set of *key‑value* vector pairs to an output vector.  
The output is computed as a weighted sum of the values, where each weight reflects how well the query matches the corresponding key (often using a scaled dot‑product similarity).  

In neural‑network models like the Transformer, attention lets each position in a sequence selectively focus on (attend to) other positions, enabling the model to capture relationships and dependencies across the entire input. Multi‑head attention extends this idea by applying several attention operations in parallel, allowing the model to learn different types of relationships simultaneously.
